# Notebook 10 — final publication settings (resume-only)

Notebook 09 is a **from-scratch** validation suite: it re-runs S1–FU2 before it ever reaches
the S10 hyperparameter sweep. Cluster run 4 hit the 12 h walltime on both conv experiments,
so `cnn_cifar` and `imagenet_cnn` came back with the expensive part done and the *decision*
missing. Re-launching nb09 would spend another ~8 h reproducing what is already in the JSON.

**This notebook resumes instead.** It reads `data/results/nb09_<EXP>.json`, works out which
sweep configurations are still missing, runs *only those*, and then applies notebook 09's
**preregistered** selection rule to the merged set. It adds no new rule.

| | nb09 | nb10 |
|---|---|---|
| sections run | S1, S2, S2b, S3, S4, S5, S7, FU1, FU2, S10, S11 | the missing S10 configs only |
| trees built (`imagenet_cnn`) | ~12 | 3 |
| wall clock (`imagenet_cnn`) | > 12 h (killed) | ~4.5 h |
| wall clock (`cnn_cifar`) | > 12 h (killed) | ~3.5 h |
| restart cost after a kill | everything | one tree (~1.3 h fixed setup) |

Three things it does that nb09 does not:

1. **Saves every fingerprint matrix to disk** (`data/results/nb10_<EXP>_fingerprints/*.npz`).
   Re-scoring a configuration — a new metric, a new control, a reviewer's question — then costs
   zero refits, ever again.
2. **Puts an error bar on the selection.** The conv sweeps separate their top configurations by
   0.001–0.03 silhouette with no spread attached, and the pretrained SqueezeNet has exactly one
   model seed, so nb09's model-seed error bar does not exist for it. §4 computes a *paired*
   stimulus-subsampling CI, which works for every experiment and answers the only question a
   reviewer will ask about a 0.36-vs-0.36 comparison.
3. **Checkpoints after every tree**, so a walltime kill costs one configuration, not a night.

The selection rule is **not** re-opened. §4's confidence intervals are *reported*, never gated
on: changing the rule after seeing the numbers is the mistake nb09 already made once and had to
disclose. The CI tells you how to *describe* the choice in the paper — "selected" versus
"selected, and the margin is inside the noise" — not which configuration to take.

```
NB10_EXP=cnn_cifar|imagenet_cnn|mlp_even_odd|mlp_digit|vit_mnist   (default imagenet_cnn)
NB10_MODE=cluster|local        NB10_JOBS=<BLAS threads, wall time only>
NB10_REFITS=<top-N configs to refit for the bootstrap, default 2>
```

## §0 · Setup — inherit notebook 09s pipeline verbatim

In [ ]:
# nb10 does NOT redefine the pipeline. It EXECs notebook 09's §0/§1/§2 cells, so every tree it
# builds comes out of the identical code path as the rows it is completing — which is the whole
# point: a config measured by nb10 has to be comparable to one measured by nb09. If nb09's
# setup changes, nb10 changes with it and the comparison stays honest.
import os, sys, json, time
sys.path.insert(0, '..')

EXP      = os.environ.get('NB10_EXP',  'imagenet_cnn')
MODE     = os.environ.get('NB10_MODE', 'cluster')
N_JOBS   = int(os.environ.get('NB10_JOBS',   '3'))    # BLAS threads — wall time only
N_REFITS = int(os.environ.get('NB10_REFITS', '2'))    # top-N configs refit for the bootstrap
B_BOOT   = int(os.environ.get('NB10_BOOT',   '1000'))
SUB_FRAC = float(os.environ.get('NB10_FRAC', '0.8'))

# nb09's §0 reads NB09_*; set them so the inherited cells build the right experiment.
os.environ['NB09_EXP'], os.environ['NB09_MODE'] = EXP, MODE

import nbformat
_NB09_PATH = '09_validation_all_models.ipynb'
_nb09  = nbformat.read(_NB09_PATH, as_version=4)
_setup = []
for _c in _nb09.cells:
    if _c.cell_type != 'code':
        continue
    _setup.append(_c.source)
    if 'build_experiment(EXP)' in _c.source:      # §2's last line — stop there
        break
else:
    raise RuntimeError('nb09 setup cells not found: no cell calls build_experiment(EXP). '
                       'Did §2 move? nb10 inherits nb09 §0-§2 and cannot run without them.')

print(f'exec-ing {len(_setup)} setup cells from {_NB09_PATH} '
      f'(this builds the base tree — the single fixed cost of a launch) …')
for _i, _src in enumerate(_setup):
    exec(compile(_src, f'{_NB09_PATH}:setup{_i}', 'exec'), globals())

# ── nb10's own outputs; everything else (REPO, RES_DIR, checkpoint, sep_metrics,
# ── stab_mean, node_arbor_pos, jsonable, savefig, …) is inherited from nb09 §0.
FIG_DIR     = os.path.join(REPO, 'figs', '10_final_settings', EXP)
F_DIR       = os.path.join(RES_DIR, f'nb10_{EXP}_fingerprints')
RESULT_PATH = os.path.join(RES_DIR, f'nb10_{EXP}.json')
for _d in (FIG_DIR, F_DIR):
    os.makedirs(_d, exist_ok=True)

SWEEP_MAX_ITER = BFT_MAX_ITER          # configs must be scored at nb09's NMF budget
results.update(notebook='10', nb10=dict(n_jobs=N_JOBS, refits=N_REFITS, b_boot=B_BOOT,
                                        sub_frac=SUB_FRAC, sweep_max_iter=SWEEP_MAX_ITER))
print(f'\nnb10  EXP={EXP}  MODE={MODE}  n_samples={n_samples}  max_iter={SWEEP_MAX_ITER}  '
      f'n_jobs={N_JOBS}\n  -> {os.path.relpath(RESULT_PATH, REPO)}')

## §1 · What is already known, and what is missing

Rebuilds notebook 09's S10 configuration grid from the FU1 rank sweep and the threshold
cost pre-pass **stored in the prior JSON**, then subtracts the rows that already exist. The
rank profiles are recomputed with nb09's own logic rather than read back, because a run that
was killed mid-sweep never wrote its `rank_profiles` block — which is exactly the case this
notebook is for.

In [ ]:
def _load_json(*paths):
    for p in paths:
        if os.path.exists(p):
            return json.load(open(p)), p
    return None, None


prev, prev_path = _load_json(os.path.join(RES_DIR, f'nb09_{EXP}.json'),
                             os.path.join(REPO, 'logs', 'results', f'nb09_{EXP}.json'))
if prev is None:
    raise RuntimeError(f'no nb09 results for {EXP}. nb10 COMPLETES a validation run; '
                       'it does not start one. Run notebook 09 first.')
print(f'prior : {os.path.relpath(prev_path, REPO)}')
print(f'        sections={prev.get("completed_sections")}  n={prev.get("config", {}).get("n_samples")}')
if prev.get('mode') != MODE:
    print(f'  WARNING: prior mode={prev.get("mode")!r} but NB10_MODE={MODE!r}. Rows from the '
          'two compute profiles are NOT comparable — the merged sweep would be meaningless.')

prior10, prior10_path = _load_json(RESULT_PATH)     # resume of an earlier nb10 launch
if prior10:
    print(f'resume: {os.path.relpath(prior10_path, REPO)} '
          f'({len(prior10.get("HP_sweep", {}).get("configs", []))} rows already scored)')

# ── rebuild nb09's config grid ───────────────────────────────────────────────
hp_prev  = prev.get('HP_sweep', {})
costs    = hp_prev.get('cost_prepass', {})
dropped  = [float(x) for x in hp_prev.get('dropped_thresholds', [])]
thr_run  = sorted(float(t) for t in costs if float(t) not in dropped)

fu1p = {int(k): v for k, v in prev.get('FU1_rank_sweep', {}).get('per_layer', {}).items()}
if not fu1p:
    raise RuntimeError('prior run has no FU1_rank_sweep — the rank profiles cannot be rebuilt.')
_lids  = sorted(fu1p)
_k_def = [fu1p[li]['default_k'] for li in _lids]

# Identical construction to nb09 S10: FU1's per-layer K at each R2 target (layers that never
# reach it keep their default), plus two direct rank probes on the outcome axes.
RANK_PROFILES = {'default': _k_def}
for _key, _tag in (('k_at_r2_090', 'K@R2.90'), ('k_at_r2_095', 'K@R2.95')):
    _p = [fu1p[li][_key] or fu1p[li]['default_k'] for li in _lids]
    if _p != _k_def and _p not in RANK_PROFILES.values():
        RANK_PROFILES[_tag] = _p
for _mult, _tag in ((0.7, 'rank x0.7'), (1.3, 'rank x1.3')):
    _p = [max(1, int(round(k * _mult))) for k in _k_def]
    if _p != _k_def and _p not in RANK_PROFILES.values():
        RANK_PROFILES[_tag] = _p

configs  = [{'name': f'thr={t}', 'stimulus_threshold': t} for t in thr_run]
configs += [{'name': f'rank={tag}', 'k_max': p}
            for tag, p in RANK_PROFILES.items() if tag != 'default' and p]

thr_default = float(ctx['bft_kwargs'].get('stimulus_threshold', 0.0))
INCUMBENT   = f'thr={thr_default}'          # == the notebook's current settings == base tree

# ── merge what exists ────────────────────────────────────────────────────────
rows, provenance = {}, {}
for _src, _d in (('nb09', prev), ('nb10', prior10 or {})):
    for r in _d.get('HP_sweep', {}).get('configs', []):
        if 'error' not in r:
            # stamp the origin so a row carried through an nb10 resume still names the
            # notebook that MEASURED it, not the file it was last written to
            r.setdefault('source', _src)
            rows[r['name']] = r
            provenance[r['name']] = r['source']

grid_names = [c['name'] for c in configs]
for _n in rows:
    if _n not in grid_names:
        print(f'  WARNING: scored row {_n!r} is not in the rebuilt grid — grid drift between '
              'nb09 and nb10. Check CANDIDATE_THR / FU1 before trusting the merge.')
missing = [c for c in configs if c['name'] not in rows]

print(f'\nrank profiles : {RANK_PROFILES}')
print(f'thresholds    : {thr_run}   (incumbent: {INCUMBENT})')
print(f'\ngrid ({len(configs)} configs):')
for c in configs:
    r = rows.get(c['name'])
    print('  %-16s %-6s %s' % (
        c['name'], provenance.get(c['name'], 'MISSING'),
        ('sil=%.4f knn=%.4f stab=%.3f dim=%3d' % (r['silhouette'], r['knn_acc'],
                                                  r['min_stability'], r['fingerprint_dim']))
        if r else '— to run —'))
print(f'\n{len(missing)} configuration(s) to run: {[c["name"] for c in missing] or "none"}')

## §2 · Run the missing configurations

Same `bft()` call, same `max_iter`, same stability estimator as nb09 S10 — the rows have to be
comparable to the ones already scored. Each finished configuration is checkpointed to disk
*and* its fingerprint matrix is saved, so a kill costs at most one tree.

The base tree built by §0 **is** the incumbent configuration (notebook 09's `thr=<default>` row
is `bft()` at the registry defaults), so its fingerprint matrix comes for free. Re-scoring it
also doubles as a reproducibility check against the prior run.

In [ ]:
def tree_labels(t, n_rows):
    """Labels for a rebuilt tree's fingerprints (verbatim from nb09 S10).

    Layer-dict mode returns BFTResult.targets as an all-zeros array rather than None, so a
    plain `is not None` check silently yields a single-class label vector (and nan metrics).
    """
    y = getattr(t, 'targets', None)
    if y is not None and len(y) == n_rows and len(np.unique(y)) >= 2:
        return np.asarray(y).astype(int)
    if len(targets) == n_rows:
        return targets
    raise ValueError(f'cannot label {n_rows} fingerprint rows for this tree')


def _fp_path(name):
    return os.path.join(F_DIR, name.replace('=', '_').replace(' ', '_').replace('/', '_') + '.npz')


def save_fingerprints(name, F, y):
    np.savez_compressed(_fp_path(name), F=np.asarray(F, dtype=np.float32), y=np.asarray(y))


def load_fingerprints(name):
    p = _fp_path(name)
    if not os.path.exists(p):
        return None
    d = np.load(p)
    return d['F'], d['y']


def score_tree(name, cfg, tree_obj=None):
    """Score one configuration exactly as nb09 S10 does; also persist its fingerprints."""
    t0 = time.perf_counter()
    tr = tree_obj if tree_obj is not None else ctx['bft_call'](
        **{k: v for k, v in cfg.items() if k != 'name'},
        max_iter=SWEEP_MAX_ITER, n_jobs=N_JOBS)
    n_rows = tr.root.img_factors.shape[0]
    F = extract_fingerprint_matrix(tr, np.arange(n_rows))
    y = tree_labels(tr, n_rows)
    sil, knn = sep_metrics(F, y)
    nbl = {}
    for nd in tr.nodes():
        nbl.setdefault(nd.layer_idx, nd)
    stab = min(float(stab_mean(node_arbor_pos(nbl[li]), nbl[li].img_factors.shape[1],
                               SWEEP_STAB_SEEDS).mean()) for li in sorted(nbl))
    vs = tr.validation_summary()
    row = {'name': name, 'config': jsonable({k: v for k, v in cfg.items() if k != 'name'}),
           'silhouette': sil, 'knn_acc': knn, 'min_stability': stab,
           'min_preact_r2': float(vs['overall']['preact_r2']['min']) if vs else float('nan'),
           'median_preact_r2': float(vs['overall']['preact_r2']['median']) if vs else float('nan'),
           'fingerprint_dim': int(F.shape[1]), 'n_samples': int(len(y)),
           'wall_s': round(time.perf_counter() - t0, 1), 'source': 'nb10'}
    save_fingerprints(name, F, y)
    print('  %-16s sil=%.4f knn=%.4f stab=%.3f medR2=%s dim=%3d  %.0fs'
          % (name, sil, knn, stab,
             'n/a' if np.isnan(row['median_preact_r2']) else '%.3f' % row['median_preact_r2'],
             F.shape[1], row['wall_s']))
    return row


def publish(rows_dict):
    """Write the merged sweep to nb10's JSON (progressively — a kill keeps what finished)."""
    results['HP_sweep'] = {'rule': hp_prev.get('rule', ''), 'cost_prepass': costs,
                           'dropped_thresholds': dropped,
                           'rank_profiles': jsonable(RANK_PROFILES),
                           'incumbent': INCUMBENT,
                           'source': {k: provenance.get(k, 'nb10') for k in rows_dict},
                           # an errored config is NOT a finished one
                           'complete': sum('error' not in r for r in rows_dict.values()) >= len(configs),
                           'configs': [rows_dict[c['name']] for c in configs
                                       if c['name'] in rows_dict]}


# ── the base tree is the incumbent: score it for free, and check it reproduces ──
if load_fingerprints(INCUMBENT) is None and INCUMBENT in grid_names:
    print(f'scoring the base tree as {INCUMBENT} (already built by §0, no refit) …')
    base_row = score_tree(INCUMBENT, {'stimulus_threshold': thr_default}, tree_obj=tree)
    old = rows.get(INCUMBENT)
    if old is not None:
        d = base_row['silhouette'] - old['silhouette']
        print(f'  reproducibility vs prior run: {old["silhouette"]:.4f} -> '
              f'{base_row["silhouette"]:.4f}  (delta {d:+.4f})'
              + ('   <-- NOT reproduced, investigate before trusting the merge' if abs(d) > 5e-3 else ''))
    else:
        rows[INCUMBENT], provenance[INCUMBENT] = base_row, 'nb10'
        missing = [c for c in missing if c['name'] != INCUMBENT]
    results['incumbent_reproduction'] = {'nb10': base_row['silhouette'],
                                         'nb09': (old or {}).get('silhouette')}

# ── run whatever is still missing ────────────────────────────────────────────
for cfg in missing:
    try:
        rows[cfg['name']] = score_tree(cfg['name'], cfg)
        provenance[cfg['name']] = 'nb10'
    except Exception as e:                       # a bad rank profile must not kill the run
        print(f'  {cfg["name"]}: FAILED — {e}')
        rows[cfg['name']] = {'name': cfg['name'], 'config': jsonable(cfg), 'error': str(e)}
    publish(rows)
    checkpoint()

publish(rows)
checkpoint('sweep')
print(f'\nsweep complete: {results["HP_sweep"]["complete"]}  ({len(rows)}/{len(configs)} configs)')

## §3 · Refit the contenders so their fingerprints are on disk

The bootstrap in §4 needs the fingerprint **matrix** of each contender, and nb09 only kept
scalars. Refit the top `NB10_REFITS` configurations by silhouette (the incumbent is already
saved from §2) and keep their matrices forever.

This is the one place nb10 spends compute on something nb09 already did. It buys the error bar
that decides whether the paper says "selected" or "selected; the margin is inside the noise" —
and it is the last time any of these configurations ever needs to be fit.

In [ ]:
scored  = [r for r in rows.values() if 'error' not in r]
ranked  = sorted(scored, key=lambda r: -r['silhouette'])
want    = list(dict.fromkeys([r['name'] for r in ranked[:N_REFITS]] + [INCUMBENT]))
by_name = {c['name']: c for c in configs}
todo    = [n for n in want if n in by_name and load_fingerprints(n) is None]

print(f'contenders: {want}')
print(f'refits needed: {todo or "none — all fingerprints already on disk"}')

for name in todo:
    row = score_tree(name, by_name[name])
    old = rows.get(name)
    if old is not None and 'error' not in old:
        # Only the silhouette is checked. min_stability is a Monte-Carlo estimate over
        # SWEEP_STAB_SEEDS NMF restarts on a random row subsample, so it moves between runs by
        # construction; a difference there is noise in the estimator, not a failed refit. The
        # prior row is kept as the record either way — the refit exists for its fingerprints.
        d = row['silhouette'] - old['silhouette']
        print(f'    reproduces {old["silhouette"]:.4f} -> {row["silhouette"]:.4f} ({d:+.4f})')
        rows[name]['refit_silhouette'] = row['silhouette']
    else:
        rows[name], provenance[name] = row, 'nb10'
    publish(rows)
    checkpoint()

checkpoint('refits')

## §4 · How big is the gap, really — paired stimulus subsampling

Every contender is scored on the **same** stimuli in the same order, so the configurations can
be compared *paired*: draw a class-stratified 80 % subsample of the stimuli, recompute every
contender's silhouette on that subsample, repeat. The percentile interval of the paired
difference is the quantity of interest — pairing removes the stimulus-set variance that
dominates the marginal spread.

**Subsampling, not resampling with replacement.** A bootstrap draw duplicates stimuli, and a
duplicate is its own nearest neighbour at distance 0, which shrinks silhouette's within-cluster
term and inflates the statistic. `frac`-subsampling without replacement has no such artifact.

This is *not* a new selection gate — §5 applies notebook 09's preregistered rule to the merged
rows regardless of what this section says. It tells you how to describe the choice, not what to
choose.

In [ ]:
FP = {}
for name in want:
    got = load_fingerprints(name)
    if got is None:
        print(f'  {name}: no fingerprints on disk — skipped')
        continue
    FP[name] = got

_ys = [y for _, y in FP.values()]
if _ys and not all(np.array_equal(_ys[0], y) for y in _ys):
    raise RuntimeError('contenders were traced on different label vectors — not paired, '
                       'the comparison below would be invalid')

boot = {}
if len(FP) >= 2:
    y0  = _ys[0]
    rng = np.random.default_rng(0)
    groups = [np.where(y0 == c)[0] for c in np.unique(y0)]
    subs = [np.sort(np.concatenate([rng.choice(g, max(2, int(round(SUB_FRAC * len(g)))),
                                               replace=False) for g in groups]))
            for _ in range(B_BOOT)]
    print(f'{B_BOOT} paired subsamples of {len(subs[0])}/{len(y0)} stimuli '
          f'({SUB_FRAC:.0%}, class-stratified) …')

    curves = {}
    for name, (F, y) in FP.items():
        U = F / (np.linalg.norm(F, axis=1, keepdims=True) + 1e-12)
        curves[name] = np.array([silhouette_score(U[s], y[s]) for s in subs])
        print('  %-16s scored' % name)

    names = list(curves)
    stack = np.stack([curves[n] for n in names])                 # (n_configs, B)
    p_best = {n: float((stack.argmax(0) == i).mean()) for i, n in enumerate(names)}
    ref = curves[INCUMBENT] if INCUMBENT in curves else curves[names[0]]
    for n in names:
        d = curves[n] - ref
        boot[n] = {'mean': float(curves[n].mean()), 'sd': float(curves[n].std()),
                   'lo': float(np.percentile(curves[n], 2.5)),
                   'hi': float(np.percentile(curves[n], 97.5)),
                   'delta_vs_incumbent': float(d.mean()),
                   'delta_lo': float(np.percentile(d, 2.5)),
                   'delta_hi': float(np.percentile(d, 97.5)),
                   'p_beats_incumbent': float((d > 0).mean()),
                   'p_best': p_best[n]}
    print(f'\npaired against the incumbent ({INCUMBENT}):')
    print('  %-16s %-22s %-24s %s' % ('config', 'silhouette [95% CI]', 'delta [95% CI]', 'P(best)'))
    for n in sorted(names, key=lambda n: -boot[n]['mean']):
        b = boot[n]
        sep = '' if (b['delta_lo'] <= 0 <= b['delta_hi']) else '  *separated*'
        print('  %-16s %.4f [%.4f, %.4f]  %+.4f [%+.4f, %+.4f]  %.2f%s'
              % (n, b['mean'], b['lo'], b['hi'], b['delta_vs_incumbent'],
                 b['delta_lo'], b['delta_hi'], b['p_best'], sep))
else:
    print('fewer than 2 contenders have fingerprints — nothing to compare')

results['bootstrap'] = {'method': f'paired class-stratified {SUB_FRAC:.0%} stimulus subsampling '
                                  f'without replacement, B={B_BOOT}; reported, never gated on',
                        'incumbent': INCUMBENT, 'per_config': jsonable(boot)}
checkpoint('bootstrap')

In [ ]:
if boot:
    order = sorted(boot, key=lambda n: boot[n]['mean'])
    ys = np.arange(len(order))
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 0.6 * len(order) + 2.4))
    ax[0].errorbar([boot[n]['mean'] for n in order], ys, fmt='o', color='#4e79a7',
                   xerr=[[boot[n]['mean'] - boot[n]['lo'] for n in order],
                         [boot[n]['hi'] - boot[n]['mean'] for n in order]], capsize=3)
    ax[0].set(title='fingerprint silhouette (95% CI)', xlabel='silhouette')
    ax[1].errorbar([boot[n]['delta_vs_incumbent'] for n in order], ys, fmt='o', color='#e15759',
                   xerr=[[boot[n]['delta_vs_incumbent'] - boot[n]['delta_lo'] for n in order],
                         [boot[n]['delta_hi'] - boot[n]['delta_vs_incumbent'] for n in order]],
                   capsize=3)
    ax[1].axvline(0, ls='--', c='gray')
    ax[1].set(title=f'paired delta vs incumbent ({INCUMBENT})', xlabel='delta silhouette')
    for a in ax:
        a.set_yticks(ys); a.set_yticklabels(order, fontsize=8); a.margins(y=0.2)
    fig.suptitle(f'nb10 — {EXP}: is the margin real?', y=1.02)
    fig.tight_layout()
    savefig(fig, 'fig_nb10_bootstrap.pdf')
    checkpoint('bootstrap')

## §5 · Apply notebook 09's rule and put an error bar on the winner

The rule, unchanged and preregistered: **among configurations whose median-node causal R² is
within 0.05 of the best observed, take the highest fingerprint silhouette; ties break toward the
smaller fingerprint.** Stability reports but never excludes. Where the architecture yields no
causal R² (layer-dict traces: ViT, ImageNet) the faithfulness gate is vacuous *by construction*
and the selection is clustering-only — say so in the paper.

Then error bars at the selected configuration: model seeds where more than one checkpoint
exists, and the §4 stimulus interval always.

In [ ]:
R2_SLACK, STAB_GATE = 0.05, 0.85

done_ = [r for r in rows.values() if 'error' not in r]
r2s   = [r['median_preact_r2'] for r in done_ if not np.isnan(r['median_preact_r2'])]
r2_best = max(r2s) if r2s else float('nan')
ok   = [r for r in done_ if np.isnan(r['median_preact_r2'])
        or r['median_preact_r2'] >= r2_best - R2_SLACK]
pool = ok or done_
best = max(pool, key=lambda r: (r['silhouette'], -r['fingerprint_dim'])) if pool else None
if best is None:
    raise RuntimeError('no configuration scored — nothing to select')
low_stab = best['min_stability'] < STAB_GATE

final = dict(ctx['bft_kwargs'])
final.update({k: v for k, v in best['config'].items()})
final.update(weighting='img_selectivity', normalization='none')

b = boot.get(best['name'], {})
results['final_hp'] = {
    'selected_config': best['name'], 'sweep_complete': results['HP_sweep']['complete'],
    'median_preact_r2': best['median_preact_r2'], 'min_preact_r2': best['min_preact_r2'],
    'min_stability': best['min_stability'], 'silhouette': best['silhouette'],
    'fingerprint_dim': best['fingerprint_dim'], 'selected_below_stab_gate': bool(low_stab),
    'gate_statistic': 'median_preact_r2', 'r2_slack': R2_SLACK,
    'best_median_preact_r2': float(r2_best), 'n_within_slack': len(ok),
    'faithfulness_gate_vacuous': not bool(r2s),
    # The point estimate is the STORED row; the interval comes from the REFIT's fingerprints.
    # In a same-profile run those agree (the refit reproduces the row) — flag it when they do
    # not, because quoting a CI around a number it was not computed from is a real trap.
    'subsample_mean': b.get('mean'),
    'ci_refit_mismatch': (None if not b else abs(b['mean'] - best['silhouette']) > 5e-3),
    'silhouette_ci': [b.get('lo'), b.get('hi')] if b else None,
    'delta_vs_incumbent_ci': [b.get('delta_lo'), b.get('delta_hi')] if b else None,
    'separated_from_incumbent': (None if not b else
        not (b['delta_lo'] <= 0 <= b['delta_hi'])),
    'bft_kwargs': jsonable(final)}

print(f'SELECTED: {best["name"]}   (rule: max silhouette within {R2_SLACK} of the best '
      f'median-node causal R2)')
print(f'  silhouette {best["silhouette"]:.4f}   dim {best["fingerprint_dim"]}   '
      f'stability {best["min_stability"]:.3f}   '
      f'medianR2 {"n/a" if np.isnan(best["median_preact_r2"]) else "%.3f" % best["median_preact_r2"]}')
if not results['HP_sweep']['complete']:
    print('  WARNING: the grid is INCOMPLETE — this selection is provisional.')
if not r2s:
    print('  NOTE: no causal R2 in this architecture (layer-dict trace) => the faithfulness '
          'gate is vacuous. Selected on clustering and stability alone.')
if low_stab:
    print(f'  WARNING: stability {best["min_stability"]:.4f} < {STAB_GATE} (reported, not excluded)')
if b:
    sep = results['final_hp']['separated_from_incumbent']
    print(f'  vs incumbent {INCUMBENT}: delta {b["delta_vs_incumbent"]:+.4f} '
          f'[{b["delta_lo"]:+.4f}, {b["delta_hi"]:+.4f}] -> '
          + ('a real margin' if sep else
             'INSIDE THE NOISE — report the choice as immaterial, not as an improvement'))
checkpoint('final_hp')

In [ ]:
# ── error bars at the SELECTED configuration (nb09's S11 measured them at the defaults) ──
sel_cfg = {k: v for k, v in best['config'].items()}
rep = {'model_seed': []}
results['seed_repeats'] = {'complete': False, 'config': best['name'], 'raw': rep}

if len(ctx['seeds']) >= 2:
    print(f'model-seed repeats at {best["name"]} ({len(ctx["seeds"])} checkpoints, '
          f'max_iter={AUX_MAX_ITER}) …')
    for s_, m_ in sorted(ctx['seeds'].items()):
        t_ = ctx['bft_call'](model=m_, max_iter=AUX_MAX_ITER, n_jobs=N_JOBS, **sel_cfg)
        n_rows = t_.root.img_factors.shape[0]
        F_ = extract_fingerprint_matrix(t_, np.arange(n_rows))
        y_ = tree_labels(t_, n_rows)
        sil_, knn_ = sep_metrics(F_, y_)
        vs_ = t_.validation_summary()
        rep['model_seed'].append({'seed': s_, 'silhouette': sil_, 'knn_acc': knn_,
            'median_preact_r2': float(vs_['overall']['preact_r2']['median']) if vs_ else None,
            'min_preact_r2': float(vs_['overall']['preact_r2']['min']) if vs_ else None})
        print('  seed %d: sil=%.4f knn=%.4f' % (s_, sil_, knn_))
        checkpoint()
    v = [r['silhouette'] for r in rep['model_seed']]
    results['seed_repeats']['model_seed_silhouette'] = {
        'mean': float(np.mean(v)), 'std': float(np.std(v)), 'n': len(v)}
    print('  model-seed silhouette: %.4f +/- %.4f (n=%d)' % (np.mean(v), np.std(v), len(v)))
else:
    print(f'model-seed repeats skipped — {len(ctx["seeds"])} checkpoint(s) on disk. '
          'The §4 stimulus interval is the only error bar available for this experiment.')

results['seed_repeats'].update(
    complete=True, raw=jsonable(rep),
    note='NMF-seed repeats are deliberately not run: nb09 measured sigma = 0.0000 over 5 seeds '
         'because bft initialises with deterministic nndsvda. Model seeds and the stimulus '
         'subsampling interval are the only informative spreads.')
checkpoint('seed_repeats')

## §6 · The answer, ready to paste

In [ ]:
NB_VARS = {
    'mlp_even_odd': ('nb01 — 01_MLP_8_4_0134.ipynb',
                     [('K_MAX_PER_LAYER', 'k_max'), ('N_BRANCHES', 'n_branches'),
                      ('STIMULUS_THRESHOLD', 'stimulus_threshold')]),
    'mlp_digit':    ('nb02 — 02_MLP_40_20_digits.ipynb',
                     [('K_MAX', 'k_max'), ('N_BRANCHES', 'n_branches'),
                      ('STIM_THRESHOLD', 'stimulus_threshold')]),
    'cnn_cifar':    ('nb03 — 03_CNN_CIFAR10.ipynb',
                     [('K_MAX', 'k_max'), ('N_BRANCHES', 'n_branches'),
                      ('POOL_METHOD', 'conv_pool_method'), ('STIM_THRESHOLD', 'stimulus_threshold')]),
    'vit_mnist':    ('nb04 — 04_ViT.ipynb',
                     [('K_MAX', 'k_max'), ('N_BRANCHES', 'n_branches'),
                      ('STIM_THRESHOLD', 'stimulus_threshold')]),
    'imagenet_cnn': ('nb05 — 05_imagenet_cnn.ipynb',
                     [('K_MAX', 'k_max'), ('N_BRANCHES', 'n_branches'),
                      ('POOL_METHOD', 'conv_pool_method'), ('STIM_THRESHOLD', 'stimulus_threshold')]),
}

title, varmap = NB_VARS[EXP]
fh = results['final_hp']
print('=' * 78)
print(f'{EXP} -> {title}')
print('=' * 78)
for var, key in varmap:
    if key in fh['bft_kwargs']:
        print(f'{var:<18}= {fh["bft_kwargs"][key]!r}')
print()
_km = fh['bft_kwargs'].get('k_max')
if isinstance(_km, list) and any(k < 2 for k in _km):
    _eff = [max(2, int(k)) for k in _km]
    print(f'# NOTE _auto_k_factorize floors k_max at 2 (src/bft.py), so this EXECUTES as {_eff}.')
    print('#      Write the floored profile into the notebook — it is what actually runs.')
print(f'# selected: {fh["selected_config"]}  |  grid complete: {fh["sweep_complete"]}')
_ci = fh.get('silhouette_ci')
print(f'# silhouette {fh["silhouette"]:.4f}'
      + ('  |  %d%% subsample mean %.4f [%.4f, %.4f]'
         % (SUB_FRAC * 100, fh['subsample_mean'], _ci[0], _ci[1])
         if _ci and _ci[0] is not None else '')
      + f'  |  fingerprint {fh["fingerprint_dim"]} dims'
      + f'  |  stability {fh["min_stability"]:.3f}')
if fh.get('ci_refit_mismatch'):
    print(f'# WARNING the refit that produced the interval scored {fh["subsample_mean"]:.4f}, '
          f'not the stored {fh["silhouette"]:.4f}. The two numbers come from different runs — '
          'quote them together only after re-running the sweep in one compute profile.')
if not np.isnan(fh['median_preact_r2']):
    print(f'# median causal preact-R2 {fh["median_preact_r2"]:.3f} '
          f'(best in grid {fh["best_median_preact_r2"]:.3f}, slack {fh["r2_slack"]})')
else:
    print('# no causal reconstruction in this architecture — faithfulness gate vacuous')
if fh.get('separated_from_incumbent') is False:
    print(f'# the margin over {INCUMBENT} is inside the noise — describe the choice as '
          'immaterial, not as an improvement')
print('=' * 78)
print(f'\nfull record : {os.path.relpath(RESULT_PATH, REPO)}')
print(f'fingerprints: {os.path.relpath(F_DIR, REPO)}/  '
      f'({len(os.listdir(F_DIR))} matrices — re-scoring never needs a refit again)')
checkpoint('DONE')

## How to run this on the cluster

`imagenet_cnn` is the one that matters: its grid is missing `rank x1.3`, so nb09's rule cannot
be applied and `final_hp` was never written. `cnn_cifar` finished its grid — running it here
only adds the error bar and the saved fingerprints.

```bash
cd /home/jb3879/Factor_Trace
git pull

mkdir -p slurm_logs
cat > run_final.sbatch <<'EOF'
#!/bin/bash
#SBATCH --job-name=bft-final
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=8
#SBATCH --mem=128G
#SBATCH --time=08:00:00
#SBATCH --array=0-1
#SBATCH --output=slurm_logs/%x_%a.out
#SBATCH --error=slurm_logs/%x_%a.err
EXPS=(imagenet_cnn cnn_cifar)
cd "$SLURM_SUBMIT_DIR"
export MPLBACKEND=Agg NB10_MODE=cluster NB10_JOBS=8
NB10_EXP="${EXPS[$SLURM_ARRAY_TASK_ID]}" \
  ./.venv/bin/python scripts/run_nb.py notebooks/10_final_settings.ipynb
EOF
sbatch run_final.sbatch
```

`scripts/run_nb.py` executes the notebook cell by cell in a plain interpreter — no kernel, no
`ipykernel`, no `nbconvert`, which is what killed earlier cluster attempts. Everything worth
keeping goes to the JSON and the log, not to an executed `.ipynb`.

`NB10_JOBS=8` raises the BLAS thread cap from nb09's 3. It changes wall time only — the
factorisation is the same computation — so use every core the job was given.

**Expected cost.** `imagenet_cnn`: base tree ≈ 1.3 h (§0, unavoidable — it is also the incumbent
row) + `rank x1.3` ≈ 1.6 h + one contender refit ≈ 1.6 h. `cnn_cifar`: base ≈ 1.2 h + two
contender refits ≈ 2.1 h, plus ~0.7 h per CIFAR model seed if `train_extra_seeds.sh cnn` has
run. Both fit inside 8 h with room to spare; nb09 needed more than 12 and did not finish.

**If it is killed again**, relaunch the identical command. §1 reads back what landed, §2/§3 skip
every configuration whose fingerprints are already on disk, and only the ~1.3 h setup repeats.

**Check it in the morning:**

```bash
python - <<'EOF'
import json, glob
for f in sorted(glob.glob('data/results/nb10_*.json')):
    d = json.load(open(f)); fh = d.get('final_hp')
    print(f"{d['experiment']:14s} sweep_complete={d['HP_sweep']['complete']}")
    if fh:
        print('   ', fh['selected_config'], fh['bft_kwargs'])
        print('    separated from incumbent:', fh.get('separated_from_incumbent'))
EOF
```

Then update `PUBLICATION_SETTINGS.md` and the `K_MAX` / `N_BRANCHES` / `STIM_THRESHOLD` block in
the matching notebook with what §6 printed.